In [96]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import gpytorch
from gpytorch.models import ExactGP
from gpytorch.means import ConstantMean
from gpytorch.kernels import (
    ScaleKernel,
    RBFKernel,
    MaternKernel,
    PeriodicKernel,
    AdditiveKernel,
    LinearKernel,
)
from gpytorch.likelihoods import GaussianLikelihood
from gpytorch.mlls import ExactMarginalLogLikelihood

In [ ]:
device = torch.device("cpu")

In [98]:
def make_kernel(idx, num_dims):
    if idx == 0:
        return MaternKernel(nu=1.5, ard_num_dims=num_dims)
    elif idx == 1:
        return MaternKernel(nu=0.5, ard_num_dims=num_dims) + PeriodicKernel()
    elif idx == 2:
        return ScaleKernel(MaternKernel(nu=2.5, ard_num_dims=num_dims))
    else:
        raise ValueError("Committee kernel idx out of range")


# --- Обёртка для одной модели ---
class GPCommitteeModel(ExactGP):
    def __init__(self, train_x, train_y, likelihood, kernel):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = ConstantMean()
        self.covar_module = kernel

    def forward(self, x):
        mean_x = self.mean_module(x)
        covar_x = self.covar_module(x)
        return gpytorch.distributions.MultivariateNormal(mean_x, covar_x)


# --- Обучаем комитет ---
def train_committee(X_train, y_train, num_models=3, num_iter=800, device="cpu"):
    committee = []
    for i in range(num_models):
        train_x = torch.tensor(X_train, dtype=torch.float32).to(device)
        train_y = torch.tensor(y_train, dtype=torch.float32).to(device)
        likelihood = GaussianLikelihood().to(device)
        kernel = make_kernel(i, X_train.shape[1])
        model = GPCommitteeModel(train_x, train_y, likelihood, kernel).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
        mll = ExactMarginalLogLikelihood(likelihood, model)
        model.train()
        likelihood.train()
        for it in range(num_iter):
            optimizer.zero_grad()
            output = model(train_x)
            loss = -mll(output, train_y)
            if torch.isnan(loss):
                print(f"NaN in model {i}, iter {it}, breaking")
                break
            loss.backward()
            optimizer.step()
        committee.append((model, likelihood))
    return committee

In [102]:
def qbc(committee, X_sample):
    # 1. Преобразуем в тензор с градиентами
    tx = torch.tensor(X_sample, dtype=torch.float32, requires_grad=True)

    preds = []
    for model, likelihood in committee:
        model.eval()
        likelihood.eval()
        x_query_t = tx.reshape(1, -1)
        mean = model(x_query_t).mean
        preds.append(mean)

    # 3. Вычисляем дисперсию предсказаний
    preds_tensor = torch.stack(preds)  # [n_models, n_samples]
    f_avg = preds_tensor.mean(dim=0)
    print("preds and avg", preds, f_avg)
    loss = torch.var(preds_tensor - f_avg, dim=0).mean()
    # 4. Вычисляем градиенты
    tx.grad = None  # Очищаем предыдущие градиенты
    loss.backward()

    gradients = (
        tx.grad.detach().numpy() if tx.grad is not None else np.zeros_like(X_sample)
    )
    return loss.item(), gradients


def NA_query_strategy(comittee, X_sample, X_pool):
    _, grads = qbc(comittee, X_sample)  # (N, M) -> N
    grads = [torch.tensor(row, dtype=torch.float32).unsqueeze(0) for row in grads]
    grads = torch.tensor(grads, dtype=torch.float32).numpy()
    sign = np.sign(grads)
    print(sign, grads)
    for ind in range(len(sign)):
        step = 0.01
        x_gen = X_sample[ind] + step * sign[ind]
        x_gen = np.clip(x_gen, 0.0, 1.0)
        dists = np.linalg.norm(X_pool - x_gen, axis=1)
        idx = np.argmin(dists)
    stds = []
    for model, likelihood in comittee:
        model.eval()
        likelihood.eval()
        X_hold = torch.tensor(X_pool[idx], dtype=torch.float32).to(device)
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            preds = model(X_hold.reshape(1, -1))
            std = preds.variance.sqrt().cpu().numpy().flatten()
            stds.append(std)
    return idx, max(std)

In [105]:
# --- 1. Загрузка данных ---
df = pd.read_csv("ourall.csv")
df = df[df['B4TOD4'] != 0.005]
df = df[df['OMEGA5'] != 15]
df = df[df['OMEGA5'] != -15]
PARAMS = [col for col in df.columns if col != "PT_LOSS"]
X = df[PARAMS].values
y = df["PT_LOSS"].values #.reshape(-1, 1)

# --- 2. min/max по параметрам ---
param_bounds = {col: (df[col].min(), df[col].max()) for col in PARAMS}

# --- 3. Нормализация данных (очень желательно для GPs) ---
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)
y_scaler = StandardScaler()
# y_scaled = y_scaler.fit_transform(y).flatten()
# --- 4. Делим на train и pool ---
np.random.seed(42)
INIT_SIZE = 10
initial_idx = np.random.choice(len(X), size=INIT_SIZE, replace=False)
X_train, y_train = X_scaled[initial_idx], y[initial_idx]
X_pool, y_pool = (
    np.delete(X_scaled, initial_idx, axis=0),
    np.delete(y, initial_idx, axis=0),
)

In [106]:
import numpy as np
import torch


# --- Основной цикл ---
N_QUERIES = 20
num_models = 3  # число моделей в комитете

for it in range(N_QUERIES):
    # 1. Обучаем комитет (пример кода train_committee см. выше)
    committee = train_committee(X_train, y_train, num_models=num_models, num_iter=1)
    # 2. Выбираем точку по максимальному разногласию градиентов (NA-QBC)
    params = []
    max_idx, max_std = 0, 0.0
    for i in range(X_train.shape[0]):
        idx, std = NA_query_strategy(committee, X_train[i], X_pool)
        print(std)
        if std > max_std:
            max_std = std
            max_idx = i
    X_train = np.vstack([X_train, X_pool[max_idx]])
    y_train = np.append(y_train, y_pool[max_idx])
    X_pool = np.delete(X_pool, max_idx, axis=0)
    y_pool = np.delete(y_pool, max_idx, axis=0)
    print(f"Step {it + 1}: added point {max_idx}, pool left: {len(X_pool)}")
    preds_list = []
    for model, likelihood in committee:
        model.eval()
        likelihood.eval()
        X_hold = torch.tensor(X_pool, dtype=torch.float32).to(device)
        with torch.no_grad(), gpytorch.settings.fast_pred_var():
            preds = model(X_hold)
            y_pred = preds.mean.cpu().numpy().flatten()
            y_true = y_pool
        preds_list.append(y_pred)
    y_pred = np.mean(preds_list, axis=0)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"Final MAE: {mae:.4f}, R2: {r2:.4f}")


preds and avg [tensor([0.3459], grad_fn=<ViewBackward0>), tensor([0.4172], grad_fn=<ViewBackward0>), tensor([0.2982], grad_fn=<ViewBackward0>)] tensor([0.3537], grad_fn=<MeanBackward1>)
[ 1. -1. -1.  1.  1. -1. -1.] [ 2.5816215e-04 -2.7715456e-04 -4.4383240e-05  1.0607936e-04
  3.9058432e-07 -4.9192462e-05 -3.0902238e-04]
0.8049807
preds and avg [tensor([0.1202], grad_fn=<ViewBackward0>), tensor([0.1188], grad_fn=<ViewBackward0>), tensor([0.1150], grad_fn=<ViewBackward0>)] tensor([0.1180], grad_fn=<MeanBackward1>)
[ 1.  1. -1.  1.  1.  1. -1.] [ 2.2311633e-07  8.0542231e-06 -1.0818767e-05  5.1462303e-06
  1.9440718e-06  4.3066229e-06 -9.5483811e-06]
0.8049807
preds and avg [tensor([0.1391], grad_fn=<ViewBackward0>), tensor([0.1421], grad_fn=<ViewBackward0>), tensor([0.1312], grad_fn=<ViewBackward0>)] tensor([0.1375], grad_fn=<MeanBackward1>)
[ 1.  1. -1. -1.  1. -1.  1.] [ 3.5686742e-05  5.6526955e-05 -1.3456171e-04 -1.4407779e-06
  8.1019403e-05 -1.6510399e-05  7.1803202e-05]
0.785341